# API Server: Voice Cloning (Tiếng Anh/Nhật giọng nhân vật + Tiếng Việt)

Notebook này đóng gói cả 2 hệ thống đã test thành công thành 1 API, expose ra ngoài qua ngrok để web app có thể gọi tới.

**Cách dùng:**
1. Bật GPU: `Runtime > Change runtime type > T4 GPU`
2. Chạy tuần tự từng bước từ trên xuống (Bước 1 → 1.5 → Restart → 1 → 1.5 → 2 → 3 → ... → cuối)
3. Ở bước cuối, bạn sẽ nhận được 1 link công khai (dạng `https://xxxx.ngrok-free.app`) — dán link này vào web app
4. **Lưu ý**: mỗi lần chạy lại notebook, link ngrok sẽ đổi — cần cập nhật lại link mới vào web app
5. Giữ tab Colab này luôn mở khi muốn API hoạt động — đóng tab là API ngừng

## Bước 1: Cài đặt thư viện chính

In [ ]:
!pip install -q coqui-tts gdown huggingface_hub vinorm underthesea unidecode
!pip install -q cutlet fugashi unidic-lite
!pip install -q fastapi uvicorn pyngrok python-multipart nest-asyncio

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.8/862.8 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 113.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 997.3/997.3 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 648.4/648.4 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 24.5 MB/s eta 0:00:00
  

## Bước 1.5: Sửa lỗi xung đột phiên bản (bắt buộc)
**QUAN TRỌNG: Sau khi chạy cell này LẦN ĐẦU TIÊN trong session, bắt buộc phải `Runtime > Restart session`, rồi chạy lại Bước 1 và Bước 1.5 một lần nữa trước khi qua Bước 2.**

In [ ]:
!pip install -q "transformers==4.57.6"
print("Đã hạ cấp transformers.")
print("Nếu đây là lần ĐẦU TIÊN chạy cell này trong session -> Restart session rồi chạy lại từ Bước 1.")
print("Nếu là lần thứ hai (sau khi đã restart) -> có thể đi tiếp Bước 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Đã hạ cấp transformers.
Nếu đây là lần ĐẦU TIÊN chạy cell này trong session -> Restart session rồi chạy lại từ Bước 1.
Nếu là lần thứ hai (sau khi đã restart) -> có thể đi tiếp Bước 2.


## Bước 2: Cấu hình danh sách giọng đọc

Mỗi giọng là 1 mục trong `VOICES`. Có 2 loại:
- `"custom"`: giọng nhân vật clone từ audio mẫu — cần `drive_file_id` (link Google Drive, xem hướng dẫn lấy `file_id` ở các bước trước)
- `"builtin"`: giọng có sẵn trong XTTS-v2 (58 giọng dựng sẵn, không cần audio mẫu) — cần `speaker_name` đúng tên trong model

**Để thêm giọng nhân vật mới**: thêm 1 mục vào `VOICES`, dán `drive_file_id` của audio mẫu mới, đặt `id` và `name` tùy ý, rồi chạy lại từ Bước 2 trở đi.

**Để thêm giọng có sẵn (builtin)**: chạy cell "Xem danh sách giọng có sẵn" bên dưới để lấy tên giọng, rồi thêm vào `VOICES`.

In [ ]:
import os
import gdown

# Mỗi giọng có thể có audio mẫu RIÊNG cho từng ngôn ngữ (khuyến nghị — chất lượng tốt hơn hẳn),
# hoặc chỉ 1 audio mẫu dùng chung cho mọi ngôn ngữ (dễ setup, chất lượng cross-lingual kém hơn).
#
# Cách dùng "drive_file_ids" (khuyến nghị khi có nhiều audio mẫu theo ngôn ngữ):
#   "drive_file_ids": {"en": "id_audio_tieng_anh", "ja": "id_audio_tieng_nhat", "vi": "id_audio_tieng_viet"}
#   -> nếu thiếu 1 ngôn ngữ nào đó, hệ thống tự dùng file "default" làm dự phòng.
#
# Cách dùng "drive_file_id" (1 file dùng chung, đơn giản — vẫn hỗ trợ, để tương thích ngược):
#   "drive_file_id": "id_audio_duy_nhat"

VOICES = [
    {
        "id": "evernight",
        "name": "evernight",
        "type": "custom",
        "drive_file_ids": {
            "default": "1_zkKSAaw6GvfMNX4PAkv2ZsFfr3qjBK8",  # dùng cho en, vi
            "ja": "1oCfd6zIMa2eWfg3HBkXv6VoMMMkHErmM",     # audio mẫu riêng cho tiếng Nhật
        },
        "languages": ["en", "ja", "vi"],
    },
    {
        "id": "phainon",
        "name": "Phainon",
        "type": "custom",
        "drive_file_ids": {
            "default": "1CBjQDOG7SYbNWapdMK0N2LxW2GSOUZTG",  # dùng cho en, vi
            "ja": "1ltq-Cp0r7jcl5SIOW-OwE_z8fKYCCvOg",     # audio mẫu riêng cho tiếng Nhật
        },
        "languages": ["en", "ja", "vi"],
    },
    {
        "id": "castorice",
        "name": "Castorice",
        "type": "custom",
        "drive_file_ids": {
            "default": "1nSOaI3kgFCbwUudpvszCIoOY_iwllv27",  # dùng cho en, vi
            "ja": "1QvVrbnyvURIr0IDvzqDIFcItvy6DEQU4",     # audio mẫu riêng cho tiếng Nhật
        },
        "languages": ["en", "ja", "vi"],
    },
    {
        "id": "aglaea",
        "name": "Aglaea",
        "type": "custom",
        "drive_file_ids": {
            "default": "1sTi1rh8ECOuPCWAgjhP2Iu3BlOWer_VX",  # dùng cho en, vi
            "ja": "1PFT2hrGJqOHkAInRU7BSqxo1edrjwgvW",     # audio mẫu riêng cho tiếng Nhật
        },
        "languages": ["en", "ja", "vi"],
    },
    # Ví dụ thêm giọng có sẵn (builtin) — xem tên giọng ở cell bên dưới sau khi model tải xong:
    # {
    #     "id": "builtin_female_1",
    #     "name": "Giọng nữ trầm ấm",
    #     "type": "builtin",
    #     "speaker_name": "Claribel Dervla",  # thay bằng tên thật lấy từ danh sách
    #     "languages": ["en", "ja"],
    # },
]

os.makedirs("voice_samples", exist_ok=True)


def _download_sample(voice_id, lang_tag, file_id):
    """Tải 1 file audio mẫu, trả về đường dẫn cục bộ."""
    sample_path = f"voice_samples/{voice_id}_{lang_tag}.wav"
    gdown.download(id=file_id, output=sample_path, quiet=False)
    if os.path.exists(sample_path) and os.path.getsize(sample_path) > 1000:
        print(f"[{voice_id}/{lang_tag}] Tải thành công: {sample_path} ({os.path.getsize(sample_path)/1024:.0f} KB)")
        return sample_path
    else:
        print(f"[{voice_id}/{lang_tag}] LỖI: tải thất bại hoặc file rỗng — kiểm tra lại file_id.")
        return None


for voice in VOICES:
    if voice["type"] != "custom":
        continue

    # sample_paths: dict {ngôn_ngữ: đường_dẫn_file} — dùng để chọn đúng audio mẫu khi generate
    voice["sample_paths"] = {}

    if "drive_file_ids" in voice:
        # Trường hợp có nhiều audio mẫu theo ngôn ngữ
        default_id = voice["drive_file_ids"].get("default")
        default_path = _download_sample(voice["id"], "default", default_id) if default_id else None

        for lang in voice["languages"]:
            if lang in voice["drive_file_ids"]:
                path = _download_sample(voice["id"], lang, voice["drive_file_ids"][lang])
                voice["sample_paths"][lang] = path or default_path
            else:
                voice["sample_paths"][lang] = default_path

        # Giữ "sample_path" (số ít) để tương thích với code cũ, trỏ về bản mặc định
        voice["sample_path"] = default_path

    else:
        # Trường hợp chỉ có 1 file dùng chung cho mọi ngôn ngữ (cách cũ)
        shared_path = _download_sample(voice["id"], "shared", voice["drive_file_id"])
        voice["sample_path"] = shared_path
        for lang in voice["languages"]:
            voice["sample_paths"][lang] = shared_path

print(f"\nTổng cộng {len(VOICES)} giọng đã cấu hình.")

Downloading...
From: https://drive.google.com/uc?id=1_zkKSAaw6GvfMNX4PAkv2ZsFfr3qjBK8
To: /content/voice_samples/evernight_default.wav
100%|██████████| 3.55M/3.55M [00:00<00:00, 23.7MB/s]


[evernight/default] Tải thành công: voice_samples/evernight_default.wav (3471 KB)


Downloading...
From: https://drive.google.com/uc?id=1oCfd6zIMa2eWfg3HBkXv6VoMMMkHErmM
To: /content/voice_samples/evernight_ja.wav
100%|██████████| 1.34M/1.34M [00:00<00:00, 11.3MB/s]


[evernight/ja] Tải thành công: voice_samples/evernight_ja.wav (1304 KB)


Downloading...
From: https://drive.google.com/uc?id=1CBjQDOG7SYbNWapdMK0N2LxW2GSOUZTG
To: /content/voice_samples/phainon_default.wav
100%|██████████| 2.10M/2.10M [00:00<00:00, 16.0MB/s]


...

[castorice/ja] Tải thành công: voice_samples/castorice_ja.wav (1380 KB)


Downloading...
From: https://drive.google.com/uc?id=1sTi1rh8ECOuPCWAgjhP2Iu3BlOWer_VX
To: /content/voice_samples/aglaea_default.wav
100%|██████████| 1.51M/1.51M [00:00<00:00, 12.0MB/s]


[aglaea/default] Tải thành công: voice_samples/aglaea_default.wav (1478 KB)


Downloading...
From: https://drive.google.com/uc?id=1PFT2hrGJqOHkAInRU7BSqxo1edrjwgvW
To: /content/voice_samples/aglaea_ja.wav
100%|██████████| 1.67M/1.67M [00:00<00:00, 12.2MB/s]

[aglaea/ja] Tải thành công: voice_samples/aglaea_ja.wav (1634 KB)

Tổng cộng 4 giọng đã cấu hình.


## Bước 2.5: Vá lỗi "Backend should be defined in the BACKENDS_MAPPING" (bắt buộc nếu gặp lỗi này)
Đây là lỗi nội bộ đã biết của thư viện `transformers` khi Colab có sẵn `tensorboard` nhưng không có TensorFlow thật — thư viện bị nhầm tưởng cần TensorFlow rồi crash lúc import. Cell dưới đây ép `transformers` bỏ qua hoàn toàn việc kiểm tra TensorFlow, cần chạy **trước khi** import bất kỳ thứ gì từ `TTS` hoặc `transformers`.

In [ ]:
import os

# Ép transformers không kiểm tra/dùng TensorFlow, tránh lỗi "BACKENDS_MAPPING. Offending backend: tf"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TORCH"] = "1"

print("Đã đặt biến môi trường để tắt kiểm tra TensorFlow trong transformers.")

Đã đặt biến môi trường để tắt kiểm tra TensorFlow trong transformers.


## Bước 3: Tải model XTTS-v2 (dùng cho tiếng Anh/Nhật, giọng nhân vật)

In [ ]:
import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang dùng: {device}")
if device == "cpu":
    print("CẢNH BÁO: Chưa bật GPU! Vào Runtime > Change runtime type > T4 GPU")

tts_en_ja = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model XTTS-v2 (Anh/Nhật) đã sẵn sàng!")

Đang dùng: cuda
 > You must confirm the following:
 | > "I have purchased a commercial license from Coqui: licensing@coqui.ai"
 | > "Otherwise, I agree to the terms of the non-commercial CPML: https://coqui.ai/cpml" - [y/n]
 | | > y


100%|██████████| 1.87G/1.87G [00:27<00:00, 67.1MiB/s]
4.37kiB [00:00, 6.51MiB/s]
361kiB [00:00, 46.3MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 71.5kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 23.0MiB/s]


Model XTTS-v2 (Anh/Nhật) đã sẵn sàng!


### (Tham khảo) Xem danh sách giọng có sẵn (builtin) trong XTTS-v2
Chạy cell này để lấy tên chính xác các giọng dựng sẵn, dùng cho mục `speaker_name` khi thêm giọng loại `"builtin"` ở Bước 2.

In [ ]:
available_speakers = tts_en_ja.speakers
if available_speakers:
    print(f"Tổng cộng {len(available_speakers)} giọng có sẵn:\n")
    for name in available_speakers:
        print(f" - {name}")
else:
    print("Không tìm thấy danh sách giọng có sẵn.")

Tổng cộng 58 giọng có sẵn:

 - Claribel Dervla
 - Daisy Studious
 - Gracie Wise
 - Tammie Ema
 - Alison Dietlinde
 - Ana Florence
 - Annmarie Nele
 - Asya Anara
 - Brenda Stern
 - Gitta Nikolina
 - Henriette Usha
 - Sofia Hellen
 - Tammy Grit
 - Tanja Adelina
 - Vjollca Johnnie
 - Andrew Chipper
 - Badr Odhiambo
 - Dionisio Schuyler
 - Royston Min
 - Viktor Eka
 - Abrahan Mack
 - Adde Michal
 - Baldur Sanjin
 - Craig Gutsy
 - Damien Black
 - Gilberto Mathias
 - Ilkin Urbano
 - Kazuhiko Atallah
 - Ludvig Milivoj
 - Suad Qasim
 - Torcull Diarmuid
 - Viktor Menelaos
 - Zacharie Aimilios
 - Nova Hogarth
 - Maja Ruoho
 - Uta Obando
 - Lidiya Szekeres
 - Chandra MacFarland
 - Szofi Granger
 - Camilla Holmström
 - Lilya Stainthorpe
 - Zofija Kendrick
 - Narelle Moon
 - Barbora MacLean
 - Alexandra Hisakawa
 - Alma María
 - Rosemary Okafor
 - Ige Behringer
 - Filip Traverse
 - Damjan Chapman
 - Wulf Carlevaro
 - Aaron Dreschner
 - Kumar Dahl
 - Eugenio Mataracı
 - Ferran Simen
 - Xavier Hayasa

## Bước 4: Tải model viXTTS (dùng cho tiếng Việt)

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

os.makedirs("model_vi", exist_ok=True)

print("Đang tải model viXTTS (có thể mất vài phút, ~1.9GB)...")
snapshot_download(
    repo_id="capleaf/viXTTS",
    repo_type="model",
    local_dir="model_vi",
)

print("Đang tải speakers_xtts.pth từ coqui/XTTS-v2...")
hf_hub_download(
    repo_id="coqui/XTTS-v2",
    filename="speakers_xtts.pth",
    local_dir="model_vi",
)

print("Đã tải xong model viXTTS!")

Đang tải model viXTTS (có thể mất vài phút, ~1.9GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

nam-cham.wav:   0%|          | 0.00/784k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

nu-nhe-nhang.wav:   0%|          | 0.00/793k [00:00<?, ?B/s]

nu-luu-loat.wav:   0%|          | 0.00/711k [00:00<?, ?B/s]

Đang tải speakers_xtts.pth từ coqui/XTTS-v2...


speakers_xtts.pth:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

Đã tải xong model viXTTS!


In [ ]:
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

config_vi = XttsConfig()
config_vi.load_json("model_vi/config.json")

tts_vi = Xtts.init_from_config(config_vi)
tts_vi.load_checkpoint(
    config_vi,
    checkpoint_dir="model_vi",
    use_deepspeed=False,
    eval=True,
)
tts_vi.to(device)
print("Model viXTTS (tiếng Việt) đã sẵn sàng!")

Model viXTTS (tiếng Việt) đã sẵn sàng!


## Bước 5: Vá thư viện để hỗ trợ tiếng Việt (bắt buộc)
Gồm: vá module `imp` giả (cho `vinorm` chạy được trên Python 3.12), hàm chuẩn hóa text tiếng Việt, và vá tokenizer để hỗ trợ `'vi'`.

In [ ]:
import sys
import types
import importlib.util

if "imp" not in sys.modules:
    fake_imp = types.ModuleType("imp")

    def find_module(name, path=None):
        spec = importlib.util.find_spec(name)
        if spec is None or spec.origin is None:
            raise ImportError(f"No module named {name}")
        module_dir = os.path.dirname(spec.origin)
        return (None, module_dir, ("", "", 0))

    fake_imp.find_module = find_module
    sys.modules["imp"] = fake_imp
    print("Đã vá module 'imp' giả (tương thích Python 3.12).")

from vinorm import TTSnorm
from underthesea import sent_tokenize

def normalize_vietnamese_text(text):
    text = (
        TTSnorm(text, unknown=False, lower=False, rule=True)
        .replace("..", ".")
        .replace("!.", "!")
        .replace("?.", "?")
        .replace(" .", ".")
        .replace(" ,", ",")
        .replace('"', "")
        .replace("'", "")
        .replace("AI", "Ây Ai")
        .replace("A.I", "Ây Ai")
    )
    return text

def calculate_keep_len(text, lang):
    if lang in ["ja", "zh-cn"]:
        return -1
    word_count = len(text.split())
    num_punct = text.count(".") + text.count("!") + text.count("?") + text.count(",")
    if word_count < 5:
        return 15000 * word_count + 2000 * num_punct
    elif word_count < 10:
        return 13000 * word_count + 2000 * num_punct
    return -1

from TTS.tts.utils.text.cleaners import collapse_whitespace, lowercase
from TTS.tts.layers.xtts.tokenizer import VoiceBpeTokenizer

tts_vi.tokenizer.char_limits["vi"] = 200

_original_preprocess_text = VoiceBpeTokenizer.preprocess_text

def _patched_preprocess_text(self, txt, lang):
    if lang == "vi":
        txt = txt.replace('"', "")
        txt = lowercase(txt)
        txt = collapse_whitespace(txt)
        return txt
    return _original_preprocess_text(self, txt, lang)

VoiceBpeTokenizer.preprocess_text = _patched_preprocess_text

print("Đã vá xong toàn bộ hệ thống hỗ trợ tiếng Việt.")

Đã vá module 'imp' giả (tương thích Python 3.12).
Đã vá xong toàn bộ hệ thống hỗ trợ tiếng Việt.


## Bước 6: Chuẩn bị sẵn giọng nhân vật (conditioning latents)
Tính trước 1 lần, dùng lại cho mọi request — giúp API phản hồi nhanh hơn thay vì tính lại mỗi lần.

In [ ]:
# Tính trước "latent" (đặc trưng giọng) cho mỗi giọng custom — dùng chung cho cả viXTTS (tiếng Việt)
# và XTTS-v2 (Anh/Nhật, vì XTTS-v2 dùng speaker_wav trực tiếp mỗi lần gọi nên không cần tính trước,
# nhưng ta vẫn tính sẵn latent viXTTS cho từng giọng ở đây).

for voice in VOICES:
    if voice["type"] != "custom":
        continue
    vi_audio_path = voice["sample_paths"].get("vi", voice["sample_path"])
    gpt_cond_latent, speaker_embedding = tts_vi.get_conditioning_latents(
        audio_path=vi_audio_path,
        gpt_cond_len=tts_vi.config.gpt_cond_len,
        max_ref_length=tts_vi.config.max_ref_len,
        sound_norm_refs=tts_vi.config.sound_norm_refs,
    )
    voice["vi_gpt_cond_latent"] = gpt_cond_latent
    voice["vi_speaker_embedding"] = speaker_embedding
    print(f"[{voice['id']}] Đã chuẩn bị xong latent tiếng Việt (dùng audio: {vi_audio_path}).")

print(f"\nHoàn tất chuẩn bị latent cho {sum(1 for v in VOICES if v['type']=='custom')} giọng custom.")

[evernight] Đã chuẩn bị xong latent tiếng Việt (dùng audio: voice_samples/evernight_default.wav).
[phainon] Đã chuẩn bị xong latent tiếng Việt (dùng audio: voice_samples/phainon_default.wav).
[castorice] Đã chuẩn bị xong latent tiếng Việt (dùng audio: voice_samples/castorice_default.wav).
[aglaea] Đã chuẩn bị xong latent tiếng Việt (dùng audio: voice_samples/aglaea_default.wav).

Hoàn tất chuẩn bị latent cho 4 giọng custom.


## Bước 7: Định nghĩa API (FastAPI)
API có 1 endpoint chính: `POST /generate` — nhận `text` + `language` (`en`, `ja`, hoặc `vi`), trả về file audio.

In [ ]:
import uuid
import torch as _torch
import torchaudio
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Voice Cloning API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

os.makedirs("generated_audio", exist_ok=True)

SUPPORTED_LANGUAGES = {"en", "ja", "vi"}
VOICES_BY_ID = {v["id"]: v for v in VOICES}


class GenerateRequest(BaseModel):
    text: str
    language: str
    voice_id: str


@app.get("/")
def health_check():
    return {"status": "ok", "message": "Voice Cloning API đang chạy"}


@app.get("/voices")
def list_voices():
    """Trả về danh sách giọng cho web app hiển thị (không kèm dữ liệu latent nặng)."""
    return [
        {"id": v["id"], "name": v["name"], "type": v["type"], "languages": v["languages"]}
        for v in VOICES
    ]


@app.post("/generate")
def generate_speech(req: GenerateRequest):
    text = req.text.strip()
    language = req.language.strip().lower()
    voice_id = req.voice_id.strip()

    if not text:
        raise HTTPException(status_code=400, detail="Text không được để trống.")
    if language not in SUPPORTED_LANGUAGES:
        raise HTTPException(status_code=400, detail=f"Ngôn ngữ '{language}' không được hỗ trợ. Chỉ hỗ trợ: {SUPPORTED_LANGUAGES}")
    if len(text) > 500:
        raise HTTPException(status_code=400, detail="Text quá dài, giới hạn 500 ký tự.")
    if voice_id not in VOICES_BY_ID:
        raise HTTPException(status_code=400, detail=f"voice_id '{voice_id}' không tồn tại. Xem danh sách tại GET /voices.")

    voice = VOICES_BY_ID[voice_id]
    if language not in voice["languages"]:
        raise HTTPException(status_code=400, detail=f"Giọng '{voice['name']}' không hỗ trợ ngôn ngữ '{language}'.")

    output_filename = f"generated_audio/{uuid.uuid4().hex}.wav"

    try:
        if language in ("en", "ja"):
            if voice["type"] == "custom":
                sample_path = voice["sample_paths"].get(language, voice["sample_path"])
                tts_en_ja.tts_to_file(
                    text=text,
                    speaker_wav=sample_path,
                    language=language,
                    file_path=output_filename,
                )
            else:  # builtin
                tts_en_ja.tts_to_file(
                    text=text,
                    speaker=voice["speaker_name"],
                    language=language,
                    file_path=output_filename,
                )
        else:
            # Tiếng Việt: chỉ hỗ trợ giọng custom (viXTTS cần audio mẫu, không có giọng builtin cho vi)
            if voice["type"] != "custom":
                raise HTTPException(status_code=400, detail="Giọng builtin hiện chưa hỗ trợ tiếng Việt.")

            normalized_text = normalize_vietnamese_text(text)
            sentences = sent_tokenize(normalized_text)

            wav_chunks = []
            for sentence in sentences:
                if sentence.strip() == "":
                    continue
                wav_chunk = tts_vi.inference(
                    text=sentence,
                    language="vi",
                    gpt_cond_latent=voice["vi_gpt_cond_latent"],
                    speaker_embedding=voice["vi_speaker_embedding"],
                    temperature=0.3,
                    length_penalty=1.0,
                    repetition_penalty=10.0,
                    top_k=30,
                    top_p=0.85,
                    enable_text_splitting=False,
                )
                keep_len = calculate_keep_len(sentence, "vi")
                wav_chunk["wav"] = wav_chunk["wav"][:keep_len]
                wav_chunks.append(_torch.tensor(wav_chunk["wav"]))

            out_wav = _torch.cat(wav_chunks, dim=0).unsqueeze(0)
            torchaudio.save(output_filename, out_wav, 24000)

        return FileResponse(output_filename, media_type="audio/wav", filename="generated.wav")

    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Lỗi khi tạo audio: {str(e)}")


print("Đã định nghĩa API xong (hỗ trợ nhiều giọng).")

Đã định nghĩa API xong (hỗ trợ nhiều giọng).


## Bước 8: Khởi động API và expose ra ngoài qua ngrok

**Cần có tài khoản ngrok miễn phí** (https://ngrok.com) để lấy authtoken.
1. Đăng ký/đăng nhập tại https://dashboard.ngrok.com/signup
2. Vào https://dashboard.ngrok.com/get-started/your-authtoken để lấy token
3. Dán token vào biến `NGROK_AUTH_TOKEN` bên dưới

In [ ]:
NGROK_AUTH_TOKEN = "3GFwuJ09XeCdF4vZeLHbKcDegEC_7W6UKdr9eXJtYUm4jRGrF"

import nest_asyncio
from pyngrok import ngrok
import uvicorn
import asyncio

nest_asyncio.apply()

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"API đang chạy công khai tại: {public_url}")
print(f"Dán link này (kèm /generate) vào web app.")
print(f"{'='*60}\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()


API đang chạy công khai tại: NgrokTunnel: "https://clutch-bouncing-wad.ngrok-free.dev" -> "http://localhost:8000"
Dán link này (kèm /generate) vào web app.



INFO:     Started server process [476]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET /voices HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:a09a:9805:fb18:0 - "GET / HTTP/1.1" 200 OK
INFO:     2402:800:634c:acb5:c47d:

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [476]


## (Tham khảo) Cách test API bằng Python, không cần web
Mở 1 tab Colab MỚI (không phải tab đang chạy uvicorn, vì nó đang bận), rồi chạy đoạn code dưới, thay `API_URL` bằng link ngrok bạn nhận được ở Bước 8.

In [ ]:
import requests

API_URL = "https://xxxx.ngrok-free.app"  # thay bằng link thật của bạn

# Xem danh sách giọng hiện có
voices_response = requests.get(f"{API_URL}/voices")
print("Danh sách giọng:", voices_response.json())

response = requests.post(
    f"{API_URL}/generate",
    json={"text": "Xin chào, đây là test API.", "language": "vi", "voice_id": "phainon"}
)

if response.status_code == 200:
    with open("test_from_api.wav", "wb") as f:
        f.write(response.content)
    print("Thành công! File đã lưu: test_from_api.wav")
else:
    print(f"Lỗi: {response.status_code} - {response.text}")

JSONDecodeError: Expecting value: line 1 column 1 (char 0)